# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


**1. LOW_CTR_HIGH_VISIBILITY → Action: UPDATE_META_TAGS**


* **Trigger:** High model probability + `position <= 10` + `impressions > 1000` + `ctr < 0.03`.


* **Playbook:** The page already wins Page 1 visibility, meaning core content relevance is strong. Rewrite the `<title>` tag and meta description to improve click appeal, and audit the live SERP for competing Featured Snippets.


**2. STRIKING_DISTANCE_DECAY → Action: CONTENT_REFRESH_AND_INTERNAL_LINKS**


* **Trigger:** High model probability + `position` between `11` and `20` + high impression volume.


* **Playbook:** The page sits on the cusp of Page 1 ("quick-win" territory). Expand topical depth, update stale statistics, and add internal anchor links from high-authority pages.


**3. SEVERE_POSITION_DROP → Action: STRUCTURAL_REWRITE_OR_CONSOLIDATE**
* **Trigger:** High model probability + `position > 20` + declining click trajectory.


* **Playbook:** Audit search intent alignment. If a newer page on the same domain is cannibalizing the query, merge the content and implement a `301` redirect.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use:**
This pipeline operates strictly as a decision-support queue. It flags pages exhibiting statistical signs of content decay or underperformance, enabling editorial teams to efficiently prioritize their weekly review pipeline.


**Limits & Blind Spots:**
Search telemetry alone cannot observe on-SERP features (e.g., AI Overviews, Featured Snippets) that absorb user clicks. A page flagged for low CTR may be suffering from SERP layout changes rather than poor content metadata.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human-in-the-Loop Rules:**
* All flagged pages must undergo editorial review before any content is altered.
* Editors must manually verify current search intent on live Google results before merging or rewriting pages.


**The No-Go List (What NOT to Automate):**
* **Do NOT automate 301 redirects:** Incorrectly mapped redirects can permanently destroy site architecture and organic authority.
* **Do NOT auto-publish title tag rewrites:** Generative AI metadata must be reviewed to ensure brand voice compliance and prevent hallucinated claims.



In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring for Data Drift:**
* We must monitor weekly CTR distributions across rank tiers. A platform-wide Google algorithm update could permanently alter expected CTR baselines (e.g., if AI Overviews push all standard blue links further down the page).


**Retrain Triggers:**
* Retrain the Random Forest model quarterly to capture shifting seasonal search behavior.
* Trigger an immediate retraining if the `Precision@50` metric drops below `0.50` on recent editorial validation batches.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Loading the dataset
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Mapping core columns dynamically
pos_col = [c for c in df.columns if 'position' in c.lower()][0]
ctr_col = [c for c in df.columns if 'ctr' in c.lower()][0]
imp_col = [c for c in df.columns if 'impression' in c.lower()][0]
id_col = 'content_hash_id' if 'content_hash_id' in df.columns else df.columns[0]

# Applying Playbook Reason Codes to generate the actionable queueing
df['reason_code'] = 'NO_ACTION'
df.loc[(df[pos_col] <= 10) & (df[imp_col] > 1000) & (df[ctr_col] < 0.03), 'reason_code'] = 'LOW_CTR_HIGH_VISIBILITY'
df.loc[(df[pos_col] > 10) & (df[pos_col] <= 20) & (df[imp_col] > 1000), 'reason_code'] = 'STRIKING_DISTANCE_DECAY'
df.loc[(df[pos_col] > 20) & (df[imp_col] > 500), 'reason_code'] = 'SEVERE_POSITION_DROP'

action_queue = df[df['reason_code'] != 'NO_ACTION'].sort_values(by=imp_col, ascending=False).copy()
action_queue['action_label'] = action_queue['reason_code'].map({
    'LOW_CTR_HIGH_VISIBILITY': 'UPDATE_META_TAGS',
    'STRIKING_DISTANCE_DECAY': 'CONTENT_REFRESH_AND_INTERNAL_LINKS',
    'SEVERE_POSITION_DROP': 'STRUCTURAL_REWRITE_OR_CONSOLIDATE'
})

# Exporting to work/outputs/ (Safe from Git via .gitignore)
out_dir = '../work/outputs' if os.path.exists('../work') else 'work/outputs'
os.makedirs(out_dir, exist_ok=True)
export_path = f'{out_dir}/ranked_action_queue.csv'
action_queue[[id_col, 'action_label', 'reason_code', imp_col, pos_col, ctr_col]].to_csv(export_path, index=False)

print(f"Playbook queue generated! {len(action_queue)} actionable pages identified.")
print(f"Exported safely to: {export_path} (Excluded from git tracking)")
display(action_queue[[id_col, 'action_label', 'reason_code', imp_col, pos_col]].head(10))

Playbook queue generated! 8442 actionable pages identified.
Exported safely to: work/outputs/ranked_action_queue.csv (Excluded from git tracking)


,content_id,action_label,reason_code,impressions_90d,avg_position
19636,content_2cb567c3c89b,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,497727,22.2
29400,content_2dba2b1f9536,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,443434,27.9
26798,content_b28d1efd668f,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,286608,26.2
23767,content_813e88069237,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,233561,26.2
26304,content_ff94c9b6b411,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,228566,27.4
15968,content_66b4046cc144,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,217415,26.6
15405,content_a023517539fe,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,214047,85.8
7445,content_c8e9d6ab9013,UPDATE_META_TAGS,LOW_CTR_HIGH_VISIBILITY,208678,9.7
19332,content_b511d4bc4ad2,STRUCTURAL_REWRITE_OR_CONSOLIDATE,SEVERE_POSITION_DROP,205915,27.9
9200,content_c5063073d048,CONTENT_REFRESH_AND_INTERNAL_LINKS,STRIKING_DISTANCE_DECAY,192205,12.5


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.